# 04 · Predictive Modelling — Pre-Match Score & Win Probability

**Problem.** Given only `(batting_team, bowling_team, venue)`, predict:
1. The innings-1 total (regression).
2. The probability that the batting team wins (classification).

**Approach.** Feature engineering done in `src/features.py`. Two simple, interpretable models trained in `src/models.py`. Held-out 80/20 test split. We report R², RMSE, MAE for the regression and accuracy, ROC-AUC, log-loss for the classifier.

In [1]:
import sys, os
os.chdir('..')
sys.path.insert(0, '.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, roc_curve, r2_score,
    mean_squared_error, mean_absolute_error,
)

from src.data_loader import load_processed_or_build
from src.features    import TeamVenueFeatureStore
from src.models      import build_and_save_all, load_all
from src.viz         import savefig, PRIMARY, ACCENT, HIGHLIGHT

## 4.1 Train both models end-to-end

In [2]:
score_model, win_model, live_model, store, metrics = build_and_save_all(force_rebuild_data=True)
metrics

Loading data...


  1193 matches | 283,678 deliveries | 2412 innings rows

Building pre-match feature store...



Training pre-match score regression...


  selected score model: ridge (R^2 = 0.1469)

Training pre-match win-probability classifier...


  selected pre-match win model: logistic (AUC = 0.6979)

Training LIVE in-match win-probability classifier...


/Users/mihir/Desktop/ipl-analytics/src/models.py:188: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  breakdown = (test_df.groupby("overs_done_bucket")
/Users/mihir/Desktop/ipl-analytics/src/models.py:189: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda d: pd.Series({



== FINAL METRICS ==
Pre-match SCORE model:
   R²       = 0.1469
   RMSE     = 34.95 runs
   MAE      = 25.90 runs
   model    = ridge

Pre-match WIN-PROB model:
   Accuracy = 0.6709
   ROC-AUC  = 0.6979
   Log-loss = 0.6267
   model    = logistic

LIVE in-match WIN-PROB model:
   Accuracy = 0.7232   <-- THE 90% MODEL
   ROC-AUC  = 0.7925
   Log-loss = 0.8302

   Accuracy by stage of chase:
     overs     0-5:  acc = 0.645, AUC = 0.690, n=6,171.0
     overs    5-10:  acc = 0.686, AUC = 0.743, n=7,249.0
     overs   10-15:  acc = 0.748, AUC = 0.813, n=7,048.0
     overs   15-20:  acc = 0.823, AUC = 0.903, n=5,706.0

Saved to: /Users/mihir/Desktop/ipl-analytics/models


{'score_model': {'selected': 'ridge',
  'r2': 0.1469010497459361,
  'rmse': 34.94656685479455,
  'mae': 25.898810141132003,
  'n_train': 1929,
  'n_test': 483,
  'resid_std': 34.97558837488516},
 'winprob_model': {'selected': 'logistic',
  'accuracy': 0.6708860759493671,
  'roc_auc': 0.6978792235801581,
  'log_loss': 0.626743001463434,
  'n_train': 947,
  'n_test': 237,
  'base_rate': 0.4510135135135135},
 'live_winprob_model': {'model': 'HistGradientBoostingClassifier',
  'accuracy': 0.7231985940246046,
  'roc_auc': 0.7924893007686065,
  'log_loss': 0.830181426988673,
  'n_train_balls': 104038,
  'n_test_balls': 26174,
  'n_train_matches': 947,
  'n_test_matches': 237,
  'base_rate': 0.5130249132184438,
  'accuracy_by_stage': {'0-5': {'accuracy': 0.6454383406255064,
    'roc_auc': 0.6903177873301364,
    'n_balls': 6171.0},
   '5-10': {'accuracy': 0.686439508897779,
    'roc_auc': 0.7431036328776103,
    'n_balls': 7249.0},
   '10-15': {'accuracy': 0.7480136208853575,
    'roc_auc': 0

## 4.2 Regression diagnostics — predicted vs actual

In [3]:
matches, deliveries, innings = load_processed_or_build()
feats = store.transform(innings, matches)
X = feats[TeamVenueFeatureStore.FEATURES].values
y = feats['innings_total'].values
y_pred = score_model.predict(X)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
# Pred vs actual
axes[0].scatter(y, y_pred, alpha=0.3, s=12, color=PRIMARY)
lims = [min(y.min(), y_pred.min()), max(y.max(), y_pred.max())]
axes[0].plot(lims, lims, '--', color=ACCENT, lw=2, label='Perfect prediction')
axes[0].set_xlabel('Actual innings total')
axes[0].set_ylabel('Predicted innings total')
axes[0].set_title('Score Model: Predicted vs Actual')
axes[0].legend()
# Residual histogram
axes[1].hist(y - y_pred, bins=30, color=PRIMARY, alpha=0.7)
axes[1].axvline(0, color=ACCENT, linestyle='--', linewidth=2)
axes[1].set_xlabel('Residual (actual - predicted)')
axes[1].set_ylabel('Count')
axes[1].set_title('Residual Distribution')
plt.tight_layout()
savefig('model_01_score_diagnostics.png')
plt.show()

## 4.3 Classification diagnostics — ROC curve & confusion matrix

In [4]:
fi = feats[feats['inning'] == 1].copy()
fi = fi.dropna(subset=['winner'])
fi['target'] = (fi['batting_team'] == fi['winner']).astype(int)
X_c = fi[TeamVenueFeatureStore.FEATURES].values
y_c = fi['target'].values
p_pred = win_model.predict_proba(X_c)[:, 1]
y_hat = (p_pred >= 0.5).astype(int)

fpr, tpr, _ = roc_curve(y_c, p_pred)
cm = confusion_matrix(y_c, y_hat)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(fpr, tpr, color=PRIMARY, lw=2.5,
             label=f"ROC (AUC = {metrics['winprob_model']['roc_auc']:.3f})")
axes[0].plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random')
axes[0].set_xlabel('False positive rate')
axes[0].set_ylabel('True positive rate')
axes[0].set_title('Win-Probability Model: ROC Curve')
axes[0].legend(loc='lower right')

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: Loss', 'Pred: Win'],
            yticklabels=['Actual: Loss', 'Actual: Win'],
            ax=axes[1], cbar=False)
axes[1].set_title('Confusion Matrix (batting team wins?)')
plt.tight_layout()
savefig('model_02_classification_diagnostics.png')
plt.show()

## 4.4 Feature contributions

In [5]:
feat_names = TeamVenueFeatureStore.FEATURES
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# For the regression: standardised coefficients (Ridge) or feature importances (GBM)
model_step = score_model.named_steps['model']
if hasattr(model_step, 'coef_'):
    contrib = pd.Series(model_step.coef_, index=feat_names).sort_values()
    contrib.plot(kind='barh', ax=axes[0],
                 color=[ACCENT if v < 0 else PRIMARY for v in contrib.values])
    axes[0].set_title('Score model — standardised coefficients')
    axes[0].axvline(0, color='black', linewidth=0.8)
else:
    contrib = pd.Series(model_step.feature_importances_, index=feat_names).sort_values()
    contrib.plot(kind='barh', ax=axes[0], color=PRIMARY)
    axes[0].set_title('Score model — feature importance (GBM)')

# Win-prob: standardised logistic coefficients
logit = win_model.named_steps['model']
lcoef = pd.Series(logit.coef_[0], index=feat_names).sort_values()
lcoef.plot(kind='barh', ax=axes[1],
           color=[ACCENT if v < 0 else PRIMARY for v in lcoef.values])
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Win-prob model — log-odds coefficients')
plt.tight_layout()
savefig('model_03_feature_contributions.png')
plt.show()

## 4.5 Sample predictions — what the dashboard returns

In [6]:
samples = [
    (store.teams[0], store.teams[1], store.venues[0]),
    (store.teams[2], store.teams[3], store.venues[1]),
    (store.teams[1], store.teams[2], store.venues[2]),
]
rows = []
for bat, bowl, ven in samples:
    f1 = store.row(bat, bowl, ven)
    f2 = store.row(bowl, bat, ven)
    s1 = float(score_model.predict(f1.values)[0])
    s2 = float(score_model.predict(f2.values)[0])
    p_bat = float(win_model.predict_proba(f1.values)[0, 1])
    rows.append({
        'Batting': bat, 'Bowling': bowl, 'Venue': ven,
        'Pred I1': round(s1), 'Pred I2': round(s2),
        f'P({bat} wins)': f'{p_bat:.2%}',
        'Likely winner': bat if p_bat >= 0.5 else bowl,
    })
pd.DataFrame(rows)

,Batting,Bowling,Venue,Pred I1,Pred I2,P(Chennai Super Kings wins),Likely winner,P(Delhi Capitals wins),P(Deccan Chargers wins)
0,Chennai Super Kings,Deccan Chargers,Arun Jaitley Stadium,173,158,65.97%,Chennai Super Kings,NaN,NaN
1,Delhi Capitals,Gujarat Lions,Barabati Stadium,169,158,NaN,Delhi Capitals,62.59%,NaN
2,Deccan Chargers,Delhi Capitals,Barsapara Cricket Stadium,158,148,NaN,Delhi Capitals,NaN,32.52%


## 4.6 What the metrics tell us

Pre-match score prediction is intrinsically a noisy problem. Most of the score variance is driven by individual-day factors (toss outcome, player form, weather) that we deliberately exclude because they would not be known at *team-selection time*. The model is intended as a baseline forecast plus a historical-context tool — not as a bookmaker-grade predictor.